In [ ]:
# 必要パッケージ
!pip install datasets==3.6.0
!pip install haystack-ai elasticsearch-haystack

In [ ]:
## データセット用意
from datasets import load_dataset
subjqa = load_dataset("subjqa", name="electronics")
import pandas as pd
dfs = {split: dset.to_pandas() for split, dset in subjqa.flatten().items()}

In [ ]:
## Elasticsearch ダウンロード
url = "https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-7.17.10-linux-x86_64.tar.gz"
!wget -nc -q {url}
!tar -xzf elasticsearch-7.17.10-linux-x86_64.tar.gz

In [ ]:
## Elasticsearch 起動
!pkill -f org.elasticsearch.bootstrap.Elasticsearch || true
!sysctl -w vm.max_map_count=262144
!chown -R daemon:daemon /content/elasticsearch-7.17.10
!su -s /bin/bash -c 'export ES_JAVA_OPTS="-Des.cgroups.hierarchy.override=/ -XX:-UseContainerSupport"; \
/content/elasticsearch-7.17.10/bin/elasticsearch \
  -Ediscovery.type=single-node \
  -Expack.security.enabled=false \
  -Expack.ml.enabled=false \
  -Ehttp.host=127.0.0.1 \
  -Etransport.host=127.0.0.1 \
  > /tmp/es.log 2>&1 &' daemon
!sleep 25
!tail -n 120 /tmp/es.log
!curl -s http://127.0.0.1:9200

In [ ]:
### 起動確認
!curl -X GET "localhost:9200/?pretty"

In [ ]:
## Retriever の用意
### ドキュメントストアのインスタンス化
from haystack_integrations.document_stores.elasticsearch import ElasticsearchDocumentStore
custom_mapping = {
    "properties": {
        "content": {"type": "text"},
        "id": {"type": "keyword"},
        "item_id": {"type": "keyword"},
        "question_id": {"type": "keyword"},
        "split": {"type": "keyword"}
    }
}
document_store = ElasticsearchDocumentStore(hosts="http://localhost:9200",
                                            index="my_document_store",
                                            custom_mapping=custom_mapping)

### Haystack でのドキュメントストアのインデックス化
from haystack.dataclasses import Document
from haystack.document_stores.types import DuplicatePolicy
import json
def safe_str(v):
    if v is None:
        return ""
    try:
        if pd.isna(v):
            return ""
    except Exception:
        pass
    if isinstance(v, (dict, list, tuple)):
        return json.dumps(v, ensure_ascii=False)
    return str(v)
for split, df in dfs.items():
    # 重複するレビュー除去
    work = df.dropna(subset=["context"]).drop_duplicates(subset="context")
    docs = [
        Document(
            content=safe_str(row.get("context")).strip(),
            meta={
                "item_id": safe_str(row.get("title")),
                "question_id": safe_str(row.get("id")),
                "split": safe_str(split),
            },
        )
        for _, row in work.iterrows()
        if safe_str(row.get("context")).strip()
    ]
    if docs:
        written = document_store.write_documents(
            docs,
            policy=DuplicatePolicy.SKIP,
            refresh="wait_for",
        )
print(f"Loaded {document_store.count_documents()} documents")

### Elastic Search を用いたレトリーバー
from haystack_integrations.components.retrievers.elasticsearch import ElasticsearchBM25Retriever
es_retriever = ElasticsearchBM25Retriever(document_store=document_store)

In [ ]:
# Retriever 評価パイプライン
from haystack import Pipeline
from haystack.components.evaluators import DocumentRecallEvaluator

class EvalRetrieverPipeline:
  def __init__(self, retriever):
    self.retriever = retriever
    self.eval_retriever = DocumentRecallEvaluator()
    pipe = Pipeline()
    pipe.add_component(component=self.retriever, name="ESRetriever", inputs=["Query"])
    pipe.add_component(component=self.eval_retriever, name="Evaluator", inputs=["ESRetriever"])
    self.pipeline = pipe

pipe = EvalRetrieverPipeline(es_retriever)